# VQE: Algoritmo Variacional Cuántico

**Objetivo:** implementar VQE desde el principio variacional y minimizar la energía de un Hamiltoniano simple.

El principio variacional: para cualquier estado |ψ(θ)⟩, E(θ) = ⟨ψ(θ)|H|ψ(θ)⟩ ≥ E_0 (energía del ground state).
Optimizando θ minimizamos E(θ) → E_0.

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Hamiltoniano simple de 2 qubits: H = -ZZ - 0.5 XX
# Eigenvalores conocidos: {-1.5, -0.5, 0.5, 1.5}
Z = np.array([[1,0],[0,-1]])
X = np.array([[0,1],[1,0]])
I = np.eye(2)

ZZ = np.kron(Z, Z)
XX = np.kron(X, X)
H  = -ZZ - 0.5 * XX

eigenvals = np.linalg.eigvalsh(H)
print('Eigenvalores:', eigenvals)
print('E_0 (ground state) =', eigenvals[0])

In [ ]:
# Ansatz: RY rotations + CNOT
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def ansatz(theta):
    qc = QuantumCircuit(2)
    qc.ry(theta[0], 0)
    qc.ry(theta[1], 1)
    qc.cx(0, 1)
    qc.ry(theta[2], 0)
    qc.ry(theta[3], 1)
    return qc

# Función de energía E(θ) = ⟨ψ(θ)|H|ψ(θ)⟩
def energy(theta):
    qc = ansatz(theta)
    sv = Statevector.from_instruction(qc).data
    return np.real(sv.conj() @ H @ sv)

# Evaluar en punto aleatorio
theta0 = np.random.uniform(0, 2*np.pi, 4)
print(f'E(θ_rand) = {energy(theta0):.4f}  (debe ser ≥ {eigenvals[0]:.4f})')

In [ ]:
# Optimización con COBYLA
theta_init = np.zeros(4)
result = minimize(energy, theta_init, method='COBYLA',
                  options={'maxiter': 500, 'rhobeg': 0.5})

print(f'E_VQE = {result.fun:.6f}')
print(f'E_0   = {eigenvals[0]:.6f}')
print(f'Error = {abs(result.fun - eigenvals[0]):.2e}')

In [ ]:
# Curva de convergencia
history = []
def energy_track(theta):
    e = energy(theta)
    history.append(e)
    return e

result2 = minimize(energy_track, np.zeros(4), method='COBYLA',
                   options={'maxiter': 200, 'rhobeg': 0.5})

import matplotlib.pyplot as plt
plt.figure(figsize=(8,3))
plt.plot(history, color='steelblue', lw=1.5)
plt.axhline(eigenvals[0], color='red', linestyle='--', label=f'E_0 = {eigenvals[0]:.4f}')
plt.xlabel('Iteración'); plt.ylabel('Energía')
plt.title('Convergencia VQE'); plt.legend()
plt.tight_layout(); plt.savefig('vqe_conv.png', dpi=80); plt.show()

## Comparativa con Diagonalización Exacta

In [ ]:
eigvals, eigvecs = np.linalg.eigh(H)
print('Energías exactas:', eigvals.round(4))
print('\nVQE encontró:', round(result.fun, 6))
print('Exacto:      ', round(eigvals[0], 6))
print(f'Precisión: {abs(result.fun - eigvals[0])/abs(eigvals[0])*100:.4f}%')

## Ejercicio

1. Cambia el Hamiltoniano a H = -ZZ + 0.3 XX - 0.1 ZI y encuentra E_0.
2. Prueba con ansatz de 1 capa (sin segunda RY) — ¿puede representar el ground state?
3. Modifica el número de parámetros y observa el efecto en la convergencia.

In [ ]:
# Tu solución
ZI = np.kron(Z, I)
IZ = np.kron(I, Z)
H2 = -ZZ + 0.3*XX - 0.1*ZI
print('E_0 exacto H2:', np.linalg.eigvalsh(H2)[0].round(6))

## Próximos pasos

- **Lab completo:** `Cuadernos/laboratorios/08_vqe_intuicion_guiada.ipynb`
- **VQE en química:** `Cuadernos/laboratorios/28_vqe_uccsd_moleculas.ipynb`
- **Siguiente guía:** `09_qaoa_paso_a_paso.ipynb`